In [1]:
import pandas as pd
from bertopic import BERTopic
import os
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

/home/onyxia/work/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = path = os.getcwd()
df = pd.read_csv(path + "/data/processed_data.csv")
partis_corres = pd.read_csv(path + "/data/partis_correspondance.csv", sep=";")
df = pd.merge(df, partis_corres, left_on='titulaire-soutien', right_on='parti')
stopwords = [x.strip() for x in open('data/stop_words.txt').readlines()]

In [ ]:
sentence_model = SentenceTransformer("dangvantuan/sentence-camembert-base")
embeddings = sentence_model.encode(df['text'])
vectorizer = CountVectorizer(max_features=200, max_df=0.95, min_df=2, strip_accents='unicode', stop_words=stopwords)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 693.38it/s, Materializing param=pooler.dense.weight]                               
CamembertModel LOAD REPORT from: dangvantuan/sentence-camembert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
topic_model = BERTopic(verbose=True, embedding_model=sentence_model , vectorizer_model=vectorizer)
année = pd.to_datetime(df['date'], yearfirst=True, format="%Y-%m-%d")
topics, probs = topic_model.fit_transform(df['text'])

2026-04-29 20:31:42,607 - BERTopic - Embedding - Transforming documents to embeddings.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 751.90it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches:   3%|▎         | 13/507 [03:39<2:19:12, 16.91s/it]


KeyboardInterrupt: 

In [4]:
topics_over_time = topic_model.topics_over_time(df['text'], année, datetime_format="%Y-%m-%d", nr_bins=20)

6it [00:07,  1.24s/it]


In [5]:
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=20)